In [3]:
import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchtyping import TensorType
from typing import List
import numpy as np
import math

In [4]:
# Normalization for the training dataset
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # flop image 50% of the time
    transforms.RandomCrop(32, padding=4), # shift image slightly
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Normalization for the test dataset
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Using the CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Loading the 10,000 image official test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# Setting the seed
torch.manual_seed(0)

100%|██████████| 170M/170M [00:01<00:00, 98.4MB/s]


In [5]:
# Vision  Transformers class
class VisionTransformer(nn.Module):
  def __init__(self, patch_size, num_heads, num_blocks, img_size, model_dim, num_classes):
    super().__init__()
    self.num_patches = (img_size//patch_size)**2
    self.transformer_blocks = nn.ModuleList()
    for _ in range(num_blocks):
        self.transformer_blocks.append(self.TransformerBlock(model_dim,num_heads))

    self.patch_embeddings = nn.Conv2d(in_channels = 3, out_channels=model_dim, kernel_size=patch_size, stride=patch_size)
    self.embedding_dropout = nn.Dropout(p=0.1)
    self.position_embeddings = nn.Parameter(torch.zeros(1, self.num_patches+1, model_dim))
    self.cls_token = nn.Parameter(torch.zeros(1,1,model_dim))
    self.final_norm = nn.LayerNorm(model_dim)
    self.classifier = nn.Linear(model_dim, num_classes)

  def forward(self, images: TensorType[float]):
    patch_embeddings = self.patch_embeddings(images)
    patch_embeddings = patch_embeddings.flatten(2)
    patch_embeddings = patch_embeddings.transpose(1,2)
    cls_token = self.cls_token.expand(images.shape[0],-1,-1)
    x = torch.cat((cls_token, patch_embeddings), dim=1)
    x = self.embedding_dropout(x + self.position_embeddings)
    for block in self.transformer_blocks:
        x = block(x)
    x = self.final_norm(x)
    output = self.classifier(x[:, 0])
    return output


  class TransformerBlock(nn.Module):
      def __init__(self, model_dim: int, num_heads: int):
        super().__init__()

        self.attention = self.MultiHeadSelfAttention(model_dim, num_heads)
        self.linear_network = self.NeuralNetwork(model_dim)
        self.first_norm = nn.LayerNorm(model_dim)
        self.second_norm = nn.LayerNorm(model_dim)


      def forward(self, embedded: TensorType[float]):
        embedded = embedded + self.attention(self.first_norm(embedded))
        embedded = embedded + self.linear_network(self.second_norm(embedded))
        return embedded

      class MultiHeadSelfAttention(nn.Module):
        def __init__(self, model_dim: int, num_heads: int):
          super().__init__()
          self.num_heads = num_heads
          self.head_size = model_dim // self.num_heads
          self.q = nn.Linear(model_dim, model_dim, bias=False)
          self.v = nn.Linear(model_dim, model_dim, bias=False)
          self.k = nn.Linear(model_dim, model_dim, bias=False)
          self.dropout = nn.Dropout(p=0.1)
          self.out_projection = nn.Linear(model_dim,model_dim)

        def forward(self, embedded: TensorType[float]):
          B, T, C = embedded.shape
          q = self.q(embedded)
          q = q.view(B, T, self.num_heads, self.head_size).transpose(1,2)

          k = self.k(embedded)
          k = k.view(B, T, self.num_heads, self.head_size).transpose(1,2)

          v = self.v(embedded)
          v = v.view(B, T, self.num_heads, self.head_size).transpose(1,2)

          scores = q @ k.transpose(-2,-1)
          scores = scores/(self.head_size**0.5)
          scores = nn.functional.softmax(scores, dim=2)
          output = self.dropout(scores) @ v
          output = output.transpose(1,2).contiguous().view(B, T, self.head_size*self.num_heads)
          output = self.out_projection(output)
          return output

      class NeuralNetwork(nn.Module):
          def __init__(self, model_dim: int):
            super().__init__()

            self.layer1 = nn.Linear(model_dim, model_dim*4)
            self.gelu = nn.GELU()
            self.dropout_1 = nn.Dropout(p=0.2)
            self.layer2 = nn.Linear(model_dim*4, model_dim)

          def forward(self, embedded: TensorType[float]):
            x = self.layer1(embedded)
            x = self.gelu(x)
            x = self.dropout_1(x)
            x = self.layer2(x)

            return x


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = VisionTransformer(
    patch_size=4,
    num_heads=4,
    num_blocks=6,
    img_size=32,
    model_dim=128,
    num_classes=10
).to(device)

In [11]:
# Training loop
def train(model, trainloader, learning_rate, epochs):
  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)
  total_steps = len(trainloader) * 10
  warmup_steps = total_steps // 10

  def lr_lambda(step):
      if step < warmup_steps:
          return (step + 1) / warmup_steps
      progress = (step - warmup_steps) / (total_steps - warmup_steps)
      return 0.5 * (1 + math.cos(math.pi * progress))

  scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
  step_count = 0

  for epoch in range(epochs):
      running_loss = 0.0
      for i, (images, labels) in enumerate(trainloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        step_count += 1
        running_loss += loss.item()
        if i % 100 == 0:
          print(f'Epoch {epoch+1}, Step {i}, Loss: {loss.item():.3f}, Avg: {running_loss/(i+1):.3f}, LR: {scheduler.get_last_lr()[0]:.6f}')

In [12]:
# Evaluation loop
def validate(model, testloader):
  model.eval()
  correct, total = 0, 0
  with torch.no_grad():
    for images, labels in testloader:
      images, labels = images.to(device), labels.to(device)
      output = model(images)
      predicted = output.argmax(dim=1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  print(f'Accuracy: {100 * correct / total:.2f}%')
  model.train()


In [13]:
def overfit_test(model, trainloader, device):
    images, labels = next(iter(trainloader))
    images, labels = images.to(device), labels.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for step in range(200):
        optimizer.zero_grad()
        outputs = model(images)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        if step % 20 == 0:
            print(f'Step {step}, Loss: {loss.item():.3f}')

overfit_test(model, trainloader, device)

Step 0, Loss: 2.438
Step 20, Loss: 0.949
Step 40, Loss: 0.120
Step 60, Loss: 0.116
Step 80, Loss: 0.064
Step 100, Loss: 0.071
Step 120, Loss: 0.022
Step 140, Loss: 0.004
Step 160, Loss: 0.003
Step 180, Loss: 0.002


In [14]:
# Reinitialize fresh model for real training
model = VisionTransformer(
    patch_size=4, num_heads=4, num_blocks=6,
    img_size=32, model_dim=128, num_classes=10
).to(device)


# training below, unchanged
train(model, trainloader, epochs=10, learning_rate=3e-4)
validate(model, testloader)

Epoch 1, Step 0, Loss: 2.401, Avg: 2.401, LR: 0.000001
Epoch 1, Step 100, Loss: 2.188, Avg: 2.227, LR: 0.000039
Epoch 1, Step 200, Loss: 2.087, Avg: 2.121, LR: 0.000077
Epoch 1, Step 300, Loss: 1.962, Avg: 2.063, LR: 0.000116
Epoch 1, Step 400, Loss: 1.976, Avg: 2.018, LR: 0.000154
Epoch 1, Step 500, Loss: 1.647, Avg: 1.988, LR: 0.000193
Epoch 1, Step 600, Loss: 1.838, Avg: 1.956, LR: 0.000231
Epoch 1, Step 700, Loss: 1.777, Avg: 1.930, LR: 0.000269
Epoch 2, Step 0, Loss: 1.571, Avg: 1.571, LR: 0.000300
Epoch 2, Step 100, Loss: 1.709, Avg: 1.709, LR: 0.000300
Epoch 2, Step 200, Loss: 1.712, Avg: 1.686, LR: 0.000299
Epoch 2, Step 300, Loss: 1.600, Avg: 1.674, LR: 0.000299
Epoch 2, Step 400, Loss: 1.573, Avg: 1.659, LR: 0.000298
Epoch 2, Step 500, Loss: 1.476, Avg: 1.645, LR: 0.000296
Epoch 2, Step 600, Loss: 1.480, Avg: 1.634, LR: 0.000295
Epoch 2, Step 700, Loss: 1.575, Avg: 1.618, LR: 0.000293
Epoch 3, Step 0, Loss: 1.551, Avg: 1.551, LR: 0.000291
Epoch 3, Step 100, Loss: 1.372, Avg: 